## Wearable Style Dataset Analysis

In [493]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Initialisation 
df = pd.read_csv("wearable_data.csv")

# Notebook display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

# Plot styling
sns.set_style("whitegrid")
sns.set_context("notebook", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

# Initial data inspection
print("\nData shape is:", df.shape)
print("\nData types are:", df.dtypes)
print("\nAmount of null:", df.isna().sum())
print("\nAmount of duplicated:", df.duplicated().sum())
print("\n Desribe data:", df.describe())

def profile_categoricals(df, max_unique=20):
    """Describe clomuns storing categorical data"""
    categorical_cols = df.select_dtypes(include=["object", "category"]).columns

    # Printing unique value quantities
    for col in categorical_cols:
        n_unique = df[col].nunique(dropna=False)
        print(f"\n{'=' * 40}")
        print(f"  {col}  ({n_unique} unique values)")
        print(f"{'=' * 40}")

        # Skipping columns with high cardinality
        if n_unique > max_unique:
            print(f"  [Skipped - too many unique values to be useful]")
        else:
            print(df[col].value_counts(dropna=False))

profile_categoricals(df)
print(df.head(10))



Data shape is: (320, 11)

Data types are: user_id                str
date                   str
timestamp              str
steps              float64
heart_rate_bpm     float64
sleep_hours        float64
sleep_quality          str
exercise_type          str
calories_burned    float64
active_minutes       int64
spo2_percent       float64
dtype: object

Amount of null: user_id              0
date                 0
timestamp            0
steps               18
heart_rate_bpm      12
sleep_hours         19
sleep_quality        0
exercise_type      165
calories_burned     10
active_minutes       0
spo2_percent        11
dtype: int64

Amount of duplicated: 16

 Desribe data:          steps  heart_rate_bpm  sleep_hours  calories_burned  active_minutes  \
count   302.00          308.00       301.00           310.00          320.00   
mean   6978.32           77.34         6.30           281.60           58.95   
std    3493.65           29.49         1.21           134.85           27.72   
m

C:\Users\Sense\AppData\Local\Temp\ipykernel_10956\899774226.py:29: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=["object", "category"]).columns


## Remove Duplicate Rows
The dataset contained 16 exact duplicate rows. These reperesent artifacts of the the wearable's data collection. Since the rows are exact copies, we can safely keep the first occurrence of each and drop the rest. This is a lossless operation.

As all duplicates are exact copies, drop_duplicates() with default settings can be used, no need for subsets.

In [494]:
# Data cleaning

def check_duplicates(data_frame):
    """Show all duplicated rows"""
    dupes = data_frame[data_frame.duplicated(keep=False)].sort_values(["user_id", "timestamp"])
    print("Total duplicated rows:", len(dupes))
    display(dupes.head(10))
    return dupes

# Dropping exact duplicates
check_duplicates(df)
df = df.drop_duplicates().reset_index(drop=True)
check_duplicates(df);

Total duplicated rows: 32


,user_id,date,timestamp,steps,heart_rate_bpm,sleep_hours,sleep_quality,exercise_type,calories_burned,active_minutes,spo2_percent
59,U001,2024-01-12,2024-01-12 12:10:00,NaN,76.90,8.62,Good,Yoga,NaN,40,98.80
157,U001,2024-01-12,2024-01-12 12:10:00,NaN,76.90,8.62,Good,Yoga,NaN,40,98.80
15,U001,2024-01-30,2024-01-30 18:55:00,3643.00,75.00,6.77,Good,NaN,163.40,31,97.10
259,U001,2024-01-30,2024-01-30 18:55:00,3643.00,75.00,6.77,Good,NaN,163.40,31,97.10
71,U001,2024-02-28,2024-02-28 11:56:00,1163.00,320.00,7.90,Excellent,NaN,53.30,15,100.20
295,U001,2024-02-28,2024-02-28 11:56:00,1163.00,320.00,7.90,Excellent,NaN,53.30,15,100.20
247,U002,2024-01-14,2024-01-14 15:35:00,6963.00,79.60,7.49,Excellent,NaN,258.50,55,NaN
275,U002,2024-01-14,2024-01-14 15:35:00,6963.00,79.60,7.49,Excellent,NaN,258.50,55,NaN
26,U002,2024-01-19,2024-01-19 08:07:00,7997.00,65.40,6.69,Poor,NaN,339.60,78,99.80
193,U002,2024-01-19,2024-01-19 08:07:00,7997.00,65.40,6.69,Poor,NaN,339.60,78,99.80


Total duplicated rows: 0


,user_id,date,timestamp,steps,heart_rate_bpm,sleep_hours,sleep_quality,exercise_type,calories_burned,active_minutes,spo2_percent


## Date-time Formatting

Due to the multiple date formats present in "date" and "timestamp", pandas converted both columns to an "object" datatype. The "timestamp" column contains both time and date information, making the "date" column redundant. 

Before dropping the "date" column, I converted it to datetime format, and checked it against the dates in "timestamp". This will reveal if the data is actually duplicated. If so, the column will be dropped, because it's fully derivable from timestamp and duplicated storage invites drift.  

I attempted to use pd.to_datetime to convert the mixed formatted dates, but ambiguous day and month combinations resulted in inconsistent conversion. This was discovered when trying to compare the "date" and "timestamp" columns. To fix this, dates were parsed explicitly via a function.

In [495]:
# Check NaN quantity before date-time parsing
print("Amount of null in timestamp column:", df["timestamp"].isna().sum())
print("Amount of null in date column:", df["date"].isna().sum())

# Parsing timestamp
def parse_timestamp(ts):
    """Parse a timestamp in one of three known formats."""
    if pd.isna(ts):
        return pd.NaT
    ts = str(ts).strip()
    
    # Format 1: ISO - "2024-01-15 14:30:00" (or "2024-01-15 14:30")
    if "-" in ts[:5]:
        return pd.to_datetime(ts, format="%Y-%m-%d %H:%M:%S", errors="coerce") \
            if len(ts) > 16 else pd.to_datetime(ts, format="%Y-%m-%d %H:%M", errors="coerce")
    
    # Format 2: UK - "15/01/2024 14:30"
    if "/" in ts:
        return pd.to_datetime(ts, format="%d/%m/%Y %H:%M", errors="coerce")
    
    # Format 3: Textual - "Jan 15, 2024 14:30"
    return pd.to_datetime(ts, format="%b %d, %Y %H:%M", errors="coerce")
df["timestamp"] = df["timestamp"].apply(parse_timestamp)

# Parsing date
def parse_date(d):
    """Parse a date in one of three known formats."""
    if pd.isna(d):
        return pd.NaT
    d = str(d).strip()
    
    # Format 1: ISO - "2024-01-15"
    if "-" in d[:5]:
        return pd.to_datetime(d, format="%Y-%m-%d", errors="coerce")
    
    # Format 2: UK - "15/01/2024"
    if "/" in d:
        return pd.to_datetime(d, format="%d/%m/%Y", errors="coerce")
    
    # Format 3: Textual - "Jan 15, 2024 14:30"
    return pd.to_datetime(d, format="%b %d, %Y", errors="coerce")
df["date"] = df["date"].apply(parse_date)

# Checking date-time parsing
print("\nAmount of null in parsed timestamp column:", df["timestamp"].isna().sum())
print("Amount of null in parsed date column:", df["date"].isna().sum())
print("\nMin timestamp:", df["timestamp"].min())
print("Max timestamp:", df["timestamp"].max())

# Verify that the date part of timestamp matches the date column
dates_match = (df["timestamp"].dt.normalize() == df["date"]).all()
print(f"\nDate components match: {dates_match}")

if dates_match:
    # Drop the date column if it is an exact match
    df = df.drop(columns="date")
    print("Dropped redundant 'date' column")
else:
    # Find out the number of mismatches and print them for manual inspection
    mismatches = (df["timestamp"].dt.normalize() == df["date"])
    print("Number of mismatches =", (len(df) - mismatches.sum()))
    print(df[["timestamp", "date"]][df["timestamp"].dt.normalize() != df["date"]])

# Check results
df.head()

Amount of null in timestamp column: 0
Amount of null in date column: 0

Amount of null in parsed timestamp column: 0
Amount of null in parsed date column: 0

Min timestamp: 2024-01-01 10:51:00
Max timestamp: 2024-02-29 20:46:00

Date components match: True
Dropped redundant 'date' column


,user_id,timestamp,steps,heart_rate_bpm,sleep_hours,sleep_quality,exercise_type,calories_burned,active_minutes,spo2_percent
0,U003,2024-02-16 21:31:00,12855.00,82.90,6.94,Good,Running,500.00,101,97.60
1,U004,2024-01-11 08:53:00,7121.00,0.00,6.26,Good,NaN,320.50,55,97.90
2,U001,2024-01-08 19:36:00,4770.00,83.30,7.41,Fair,NaN,138.40,47,97.10
3,U002,2024-01-06 11:21:00,12956.00,67.20,6.56,Excellent,Yoga,500.10,104,95.70
4,U002,2024-02-08 13:07:00,14454.00,62.70,5.98,Good,Cycling,585.50,127,96.90


## Data Type Formatting

Data types are seemingly correct, however the "steps" column is a float64 despite containing seemingly no decimal values. Pandas has likely converted it due to int64 not being able to handle the NaN values present seen from initial data inspection. As the step counts are only whole numbers, an integer datatype is more suitable.

In [496]:
# Re-inspect database datatypes
print("Data types are:", df.dtypes)

# Checking to see if any values in "steps" contains decimal values
for step_count in df["steps"]:
    if pd.isna(step_count):
        continue
    elif step_count % 1 != 0.0:
        print(f"'steps' contains decimal values:", step_count)
        break

# Set steps to an Int64 - it contains NaN data so must be
# Int64 rather than int64
df["steps"] = df["steps"].astype("Int64")
print(f"\n'steps' is now an {df["steps"].dtype} data type")

Data types are: user_id                       str
timestamp          datetime64[us]
steps                     float64
heart_rate_bpm            float64
sleep_hours               float64
sleep_quality                 str
exercise_type                 str
calories_burned           float64
active_minutes              int64
spo2_percent              float64
dtype: object

'steps' is now an Int64 data type


## Impossible Data

Columns containing continous data such as: "steps", "heart_rate_bpm", "sleep_hours", "calories_burned", and "active_minutes" will be analysed to see if any entries contain impossible data (e.g. a heart rate of 1000). 

Firstly, biological and logical boundaries will be set at the most extreme minimum and maximum values recored for each variable. Secondly, any outliers will be handled via pairwise deletion rather than listwise deletion to preserve the size of the data set. Missingness and imputation will be considered later.

In [497]:
# A negative step count is impossible, while the upper bound 
# reflects the 24 hour step world record by Jesse Castaneda.
steps_bounds = [0, 285000]
# A resting heart rate of below 60 beats per minute (BPM) is 
# classified as bradycardia, with 26 BPM being the official
# lowest ever recorded resting heart rate by Martin Brady. 220 BPM 
# is the standard theoretical limit used when calculating maximum 
# heart rate.
heart_rate_bounds = [26, 220]
# It is impossible to sleep less than 0 hours or more than 24.
sleep_bounds = [0, 24]
# Ultramarathon runner Yiannis Kouros burned approximately 19,00
# in 24 hours. It is impossible to burn less than 0 calories.
calorie_bounds = [0, 19000]
# Similarly to sleep, activity is a time based variable. This 
# means it cannot exceed 1440 minutes (24h) or go lower than 0.
activity_bounds = [0, 1440]

def check_bounds(col, df, bounds):
    """Assess impossible data entries, and replace them with NaN"""
    # Assess entries
    mask = (df[col] < bounds[0]) | (df[col] > bounds[1])
    impossible_vals = df.loc[mask, col].copy()
    print(f"\nThere are {len(impossible_vals)} impossible values in {col}\n", impossible_vals)
    # Replace impossible values 
    df[col] = df[col].mask(mask, np.nan)
    if len(impossible_vals) > 0:
        # Check if replacement was successful
        mask = (df[col] < bounds[0]) | (df[col] > bounds[1])
        impossible_vals = df.loc[mask, col].copy()
        print(f"\nThere are now {len(impossible_vals)} impossible values in {col}\n", impossible_vals)

check_bounds("steps", df, steps_bounds)
check_bounds("heart_rate_bpm", df, heart_rate_bounds)
check_bounds("sleep_hours", df, sleep_bounds)
check_bounds("calories_burned", df, calorie_bounds)
check_bounds("active_minutes", df, activity_bounds)


There are 5 impossible values in steps
 27     -500
33     -500
104    -500
164    -500
286    -500
Name: steps, dtype: Int64

There are now 0 impossible values in steps
 Series([], Name: steps, dtype: Int64)

There are 5 impossible values in heart_rate_bpm
 1       0.00
70    320.00
130     0.00
138   320.00
298   320.00
Name: heart_rate_bpm, dtype: float64

There are now 0 impossible values in heart_rate_bpm
 Series([], Name: heart_rate_bpm, dtype: float64)

There are 0 impossible values in sleep_hours
 Series([], Name: sleep_hours, dtype: float64)

There are 5 impossible values in calories_burned
 13    -32.10
74    -16.60
93    -13.20
99     -7.70
204   -14.60
Name: calories_burned, dtype: float64

There are now 0 impossible values in calories_burned
 Series([], Name: calories_burned, dtype: float64)

There are 0 impossible values in active_minutes
 Series([], Name: active_minutes, dtype: int64)


## Handling Missing Values

Columns "exercise_type", "steps", "sleep_hours", "heart_rate_bpm", "calories_burned", and "spo2_percent" all have missing data. While most missingness likely represents device errors, NaN actually represent a key value in "exercise_type". A day with no exercise is currently reperesented by NaN, where 'rest' would be a more descriptive way to store the information. Therefore, NaN will be imputed in the "exercise type" column.

Pandas effectively handles NaN during analyses, so deleting whole entries due to a missing value is unecessary. However, the amount of NaNs in a single entry will be assessed, as multiple missing entries might signify low quality data collection and warrant listwise deletion. This is irrelevant as the data is fabricated, but has been analysed to demonstrate critical thinking.

In [498]:
# Assess missingness of current dataset
missing = df.isna().sum().to_frame(name="missing_count")
missing["missing_pct"] = (missing["missing_count"] / len(df) * 100).round(2)
print(missing.sort_values("missing_count", ascending=False))

# Replace "exercise_type" NaN values with "Rest"
exercise_type_mask = df["exercise_type"].isna()
df["exercise_type"] = df["exercise_type"].mask(exercise_type_mask, "Rest")
print("\nExercise type column post imputation:\n", df["exercise_type"].head())

# Assess missingness of individual entries
def check_data(df, entry):
    """Check how many NaN values in a single entry"""
    missing_amount = df.iloc[entry].isna().sum()
    if missing_amount > 1:
        return entry, missing_amount
    return None

# Display the index and quantity of NaN values for high missingness entries
low_qual_rows = []

for i in range(len(df)):
    result = check_data(df, i)
    if result is not None:
        low_qual_rows.append(result)

low_qual_data = pd.DataFrame(low_qual_rows, columns=["index", "missing"])
display(low_qual_data.sort_values("missing", ascending=False))

# Inspect suspected low quality data
print(df.loc[df.index.isin(low_qual_data["index"])])



                 missing_count  missing_pct
exercise_type              154        50.66
steps                       20         6.58
sleep_hours                 19         6.25
heart_rate_bpm              17         5.59
calories_burned             14         4.61
spo2_percent                 9         2.96
timestamp                    0         0.00
user_id                      0         0.00
sleep_quality                0         0.00
active_minutes               0         0.00

Exercise type column post imputation:
 0    Running
1       Rest
2       Rest
3       Yoga
4    Cycling
Name: exercise_type, dtype: str


,index,missing
0,33,3
6,208,3
1,58,2
3,107,2
2,81,2
4,112,2
5,164,2
7,217,2
8,223,2


    user_id           timestamp  steps  heart_rate_bpm  sleep_hours  \
33     U003 2024-02-25 11:08:00   <NA>           85.20          NaN   
58     U001 2024-01-12 12:10:00   <NA>           76.90         8.62   
81     U005 2024-01-29 08:01:00   7542             NaN          NaN   
107    U005 2024-01-11 11:14:00   7813           82.10          NaN   
112    U003 2024-01-15 15:24:00   <NA>           76.20          NaN   
164    U004 2024-01-24 13:51:00   <NA>             NaN         6.90   
208    U002 2024-02-10 21:49:00   <NA>           67.70          NaN   
217    U003 2024-02-08 08:41:00   5297             NaN          NaN   
223    U002 2024-01-11 20:04:00   <NA>           67.00          NaN   

    sleep_quality exercise_type  calories_burned  active_minutes  spo2_percent  
33           Poor          Rest           407.30              89           NaN  
58           Good          Yoga              NaN              40         98.80  
81           Good          Rest           296.

## Cleaning "exercise_type"

This column has mixed-case labels. This will be fixed to improve consistency in the column.

In [501]:
# Inspect column values
print(df["exercise_type"].value_counts())

# Replace mixed case entries
df["exercise_type"] = df["exercise_type"].str.title()

# Re-inspect column values
print("\n", df["exercise_type"].value_counts())

exercise_type
Rest        154
Strength     37
Running      34
Yoga         31
Walking      29
Cycling      19
Name: count, dtype: int64

 exercise_type
Rest        154
Strength     37
Running      34
Yoga         31
Walking      29
Cycling      19
Name: count, dtype: int64


## Feature Engineering

Added "day_of_week" and "is_weekend" to allow for analysis into activity schedule. Additionally, two categorical columns looking at sleep and activity were added, using evidence based bins to sort entries. 

In [525]:
# Time based features 
df["day_of_week"] = df["timestamp"].dt.day_name()
df["is_weekend"] = df["day_of_week"].isin(["Saturday", "Sunday"])

# Acticity level categorisation 
df["activity_level"] = pd.cut(
    df["steps"],
    # Classification from PMID: 14715035
    bins=[-1, 5000, 10000, 12500, np.inf],
    labels=["Sedentary", "Somewhat Active", "Active", "Highly Active"]
)

# Sleep categorisation
df["sleep_level"] = pd.cut(
    df["sleep_hours"],
    # Classification from doi: 10.1001/jamanetworkopen.2020.5246
    bins=[-1, 6, 7, 8, 9, np.inf],
    labels=["Very Short", "Short", "Medium", "Long", "Very Long"]
)

# Check featre engineering was successful
print("\nNew dataframe is:")
print(df.info())
display(df.head())


New dataframe is:
<class 'pandas.DataFrame'>
RangeIndex: 304 entries, 0 to 303
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   user_id          304 non-null    str           
 1   timestamp        304 non-null    datetime64[us]
 2   steps            284 non-null    Int64         
 3   heart_rate_bpm   287 non-null    float64       
 4   sleep_hours      285 non-null    float64       
 5   sleep_quality    304 non-null    str           
 6   exercise_type    304 non-null    str           
 7   calories_burned  290 non-null    float64       
 8   active_minutes   304 non-null    int64         
 9   spo2_percent     295 non-null    float64       
 10  day_of_week      304 non-null    str           
 11  is_weekend       304 non-null    bool          
 12  activity_level   284 non-null    category      
 13  sleep_level      285 non-null    category      
dtypes: Int64(1), bool(1), category(2),

,user_id,timestamp,steps,heart_rate_bpm,sleep_hours,sleep_quality,exercise_type,calories_burned,active_minutes,spo2_percent,day_of_week,is_weekend,activity_level,sleep_level
0,U003,2024-02-16 21:31:00,12855,82.90,6.94,Good,Running,500.00,101,97.60,Friday,False,Highly Active,Short
1,U004,2024-01-11 08:53:00,7121,NaN,6.26,Good,Rest,320.50,55,97.90,Thursday,False,Somewhat Active,Short
2,U001,2024-01-08 19:36:00,4770,83.30,7.41,Fair,Rest,138.40,47,97.10,Monday,False,Sedentary,Medium
3,U002,2024-01-06 11:21:00,12956,67.20,6.56,Excellent,Yoga,500.10,104,95.70,Saturday,True,Highly Active,Short
4,U002,2024-02-08 13:07:00,14454,62.70,5.98,Good,Cycling,585.50,127,96.90,Thursday,False,Highly Active,Very Short
